# Dialforge Sales Hard Acceptance v3.1 — Kaggle

This is the current **marketing gate** for Dialforge. It runs all three shipping Qwen tiers against the harder adversarial calibration + unseen holdout suite.

**Before Run All:**
1. Kaggle notebook Settings → Accelerator → select **any NVIDIA GPU**. One T4 is enough; T4 x2 is not required.
2. Turn **Internet ON**.
3. Click **Run All**.

Marketing passes only if **every model scores at least 85**. The notebook does not weaken the threshold if a smaller model struggles.


In [ ]:
import os, pathlib, shutil, subprocess, sys, time, json

ROOT = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path.cwd()
REPO = ROOT / 'Axemetric-Caller-Beta-Runtime'
OUT = ROOT / 'dialforge-hard-sales-v3_1'

print('=== DIALFORGE KAGGLE HARD SALES ACCEPTANCE v3.1 ===')

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('No NVIDIA GPU detected. In Kaggle Settings, enable any GPU accelerator. One T4 is sufficient.')
print('GPU(s):\n' + gpu.stdout.strip())
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['OLLAMA_NUM_PARALLEL'] = '1'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
os.environ['OLLAMA_KEEP_ALIVE'] = '30m'

# Verify internet before doing any heavy work.
probe = subprocess.run(['curl', '-I', '-L', '--max-time', '15', 'https://github.com'], capture_output=True, text=True)
if probe.returncode != 0:
    raise RuntimeError('Internet appears disabled. Turn Internet ON in Kaggle notebook settings and rerun.')

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git', str(REPO)], check=True)
sha = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Pinned source:', sha)

if shutil.which('ollama') is None:
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

log_path = ROOT / 'ollama-hard-sales.log'
log = open(log_path, 'w')
ollama = subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())

import requests
for _ in range(90):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok:
            break
    except Exception:
        pass
    time.sleep(1)
else:
    log.flush()
    raise RuntimeError('Ollama did not become ready. Log: ' + str(log_path))

print('Ollama ready. Running all three model tiers...')


In [ ]:
import subprocess, json, pathlib, pandas as pd, os, sys

OUT.mkdir(parents=True, exist_ok=True)
runner = REPO / 'benchmarks' / 'dialforge_sales_hard_acceptance_v3_1.py'
cmd = [sys.executable, str(runner), '--output-dir', str(OUT)]
print('Running:', ' '.join(cmd))
proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
lines = []
for line in proc.stdout:
    print(line, end='')
    lines.append(line)
code = proc.wait()
(OUT / 'kaggle-run.log').write_text(''.join(lines), encoding='utf-8')

report_path = OUT / 'dialforge-sales-hard-v3.json'
if not report_path.exists():
    raise RuntimeError(f'No report was produced. Exit code {code}. Check {OUT / "kaggle-run.log"}')

report = json.loads(report_path.read_text(encoding='utf-8'))
rows = []
for model, data in report['models'].items():
    rows.append({
        'Model': model,
        'Overall': data['score'],
        'Sales': data['sales']['score'],
        'Tools': data['tools']['accuracy'],
        'Integrity': data['sales']['integrity'],
        'Critical failures': data['sales']['critical_failures'],
        'Pass >=85': float(data['score']) >= 85.0,
    })
df = pd.DataFrame(rows)
display(df)

ready = all(float(x['score']) >= 85.0 for x in report['models'].values())
print('\n' + '='*72)
print('MARKETING GATE:', 'PASS' if ready else 'BLOCKED')
print('='*72)
print('Report:', report_path)
print('Log:', OUT / 'kaggle-run.log')

if not ready:
    print('Marketing stays blocked. Send me this table and the JSON/report and I will optimize the failing model(s) against the actual holdout failures.')
else:
    print('All shipping models cleared 85 on the hard adversarial gate.')
